# 02 — The "naive" assumption

*Module 02 · Naive Bayes — notebook 2 of 5*

In notebook 1 you built Bayes' rule on a *single* feature. Real penguins carry several measurements at
once — so how do we use them *together*? The answer is the one assumption that gives Naive Bayes both
its name and its power. We will see exactly what it claims, watch it be **false** on our penguins, and
then meet the surprise at the heart of the method: the classifier is often excellent anyway.

**Prerequisites:** notebook 01 (prior, likelihood, posterior, the argmax rule); module 00 notebook 02
(the two features and the feature space).

**What you will be able to do**

- State the **naive** assumption — conditional independence — in one line.
- Factorise a two-feature likelihood into a product of one-feature pieces.
- **Measure** how badly the assumption is violated (within-class correlation).
- Explain why the *decision* can stay right even when the assumption is wrong.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB

from ml_course import datasets, viz
from ml_course.colors import CLASS_CYCLE, CMAP_COUNT, CMAP_DIVERGING, COLORS

np.random.seed(0)
viz.use_course_style()

df = datasets.load_penguins()
X, y = datasets.penguins_xy(df)
print(f"rows: {df.shape[0]}   features: {list(X.columns)}   classes: {sorted(y.unique())}")
df.head()

## A recap, and a harder question

Notebook 1 answered "how probable is each species?" from **one** measurement, the bill length:
posterior ∝ prior × likelihood. But a penguin is more than its bill. To use the bill length **and** the
flipper length together, we need their **joint likelihood** — the probability of seeing *both*
measurements at once, given a species: P(bill, flipper | species). "Joint" means *both together*. The
trouble is getting it.

## Why the joint is expensive

In notebook 1 we estimated a likelihood by counting penguins in bins. We could do the same in **two**
dimensions — a grid of (bill bin × flipper bin) cells — and count. But look at what happens: with 5 bins
per feature that is 25 cells; with a third feature, 125; with ten features, nearly ten million. The
penguins spread thin, most cells end up nearly empty, and the counts turn unreliable. This is the same
curse of dimensionality that sank k-NN (module 01). We need a way to avoid filling the whole grid.

In [ ]:
# To use both features we would need the JOINT counts: a 2-D grid of (bill bin, flipper bin).
n_bins = 5
bill_edges = np.linspace(df["bill_length_mm"].min(), df["bill_length_mm"].max(), n_bins + 1)
flip_edges = np.linspace(df["flipper_length_mm"].min(), df["flipper_length_mm"].max(), n_bins + 1)

gentoo = df[df["species"] == "Gentoo"]
# Full grid of counts; empty cells are kept as 0 so the sparsity is visible.
joint_counts, _, _ = np.histogram2d(
    gentoo["bill_length_mm"], gentoo["flipper_length_mm"], bins=[bill_edges, flip_edges]
)
joint_counts = joint_counts.astype(int)
grid = pd.DataFrame(
    joint_counts,
    index=[f"bill {i + 1}" for i in range(n_bins)],
    columns=[f"flip {j + 1}" for j in range(n_bins)],
)
n_empty = int((joint_counts == 0).sum())
print(f"{len(gentoo)} Gentoo over a {n_bins ** 2}-cell grid; {n_empty} cells are empty:")
print(grid)

**Read the table.** Those 123 Gentoo are scattered across a 25-cell grid — many cells hold only a
handful of birds, and 18 of the 25 are empty. Estimating a reliable probability for *every* cell would
take far more data than we have, and it only worsens with more features. The full joint is too expensive
to count directly.

## The naive assumption: treat the features as independent

Here is the shortcut that names the method. **Assume the two features are independent *given the
species*** — that once you know a penguin is a Gentoo, its bill length tells you nothing more about its
flipper length. Under that assumption the joint **factorises** into a product of the one-feature
likelihoods we already know how to compute:

$$ P(x_1, x_2 \mid y) = P(x_1 \mid y)\, P(x_2 \mid y) $$

This is **conditional independence** — independence *within* each class. Its payoff is immediate: instead
of a full 2-D grid we need only the two **1-D marginals**, one per feature. Let us see how good that
approximation is.

In [ ]:
# Real joint P(bill, flipper | Gentoo): the 2-D counts, normalised to sum to 1.
real = joint_counts / joint_counts.sum()

# Naive assumption: the product of the two 1-D marginals (independence given the class).
p_bill = real.sum(axis=1)          # P(bill bin | Gentoo)
p_flip = real.sum(axis=0)          # P(flipper bin | Gentoo)
naive = np.outer(p_bill, p_flip)
diff = real - naive

vmax = max(real.max(), naive.max())
dmax = np.abs(diff).max()
panels = [
    (real, "real joint  P(bill, flipper | Gentoo)", CMAP_COUNT, 0.0, vmax),
    (naive, "naive  P(bill|G) x P(flipper|G)", CMAP_COUNT, 0.0, vmax),
    (diff, "difference  (real - naive)", CMAP_DIVERGING, -dmax, dmax),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (matrix, title, cmap, lo, hi) in zip(axes, panels, strict=True):
    im = ax.imshow(matrix, origin="lower", cmap=cmap, aspect="auto", vmin=lo, vmax=hi)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("flipper bin")
    ax.set_ylabel("bill bin")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()
print(f"largest gap between real and naive: {dmax:.3f}")

**Read the figure.** Left is the **real** joint for Gentoo — the mass lines up along a **diagonal**:
Gentoo with longer bills tend to have longer flippers. The middle panel is the **naive** version, the
product of the two marginals — it spreads the same mass into a **rectangular** block, with no diagonal.
The right panel is their **difference** (real − naive): teal where the real data has *more* mass than the
product predicts (along the diagonal), coral where it has *less* (the off-diagonal corners). That
difference **is** the correlation the naive assumption throws away. The two are not equal — the
assumption is an approximation.

## Where it breaks — correlation within a class

That diagonal ridge has a name: **correlation**. Within a single species, the two measurements move
together. Let us look at the raw cloud and put a number on it.

In [ ]:
f1, f2 = "bill_length_mm", "flipper_length_mm"

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for i, sp in enumerate(sorted(y.unique())):
    m = df["species"] == sp
    ax.scatter(
        df.loc[m, f1], df.loc[m, f2],
        color=CLASS_CYCLE[i], edgecolor=COLORS["text"], linewidth=0.4, s=40, alpha=0.85, label=sp,
    )
ax.set_xlabel(f1)
ax.set_ylabel(f2)
ax.set_title("Each species' cloud is tilted — the two features are correlated")
ax.legend()
plt.show()

print(f"overall correlation (both species pooled): {df[f1].corr(df[f2]):.3f}")
for sp in sorted(y.unique()):
    s = df[df["species"] == sp]
    print(f"within-class correlation, {sp:7}: {s[f1].corr(s[f2]):.3f}")

**Read the figure.** Each species forms its own cloud, and each cloud is **tilted**: within the
Adélie and within the Gentoo, longer bills go with longer flippers. The numbers confirm it — the
within-class correlation is 0.33 for Adélie and 0.66 for Gentoo. The naive assumption says these should
be **zero** (no tilt within a class), so it is violated here — moderately. One honest caution: the
*overall* correlation, pooling both species, is 0.87 — but most of that is the gap *between* the clouds,
not the tilt inside them. The assumption is about the **within-class** number, and that is what we
measured.

## What the assumption buys

Why make an assumption we can see is false? Because it is extraordinarily cheap. For d features, the full
joint grid needs bins raised to the power d — exponential, hopeless past a few features. The naive product
needs only **d** one-dimensional marginals — linear. That is what lets Naive Bayes run on thousands of
features (whole vocabularies of words, in notebook 5) where counting a joint grid, or measuring k-NN
distances, would drown. The assumption trades a little fidelity for an enormous saving.

## The real question: does the decision break?

So the assumption is false, and the naive product gets the joint *probability* wrong. The question that
matters for a classifier is different: does it get the *decision* wrong? To find out, we compare three
models under cross-validation — Naive Bayes (which ignores the correlation) against two cousins that
**use** it: QDA (a full covariance per class) and LDA (one shared covariance). If the correlation
mattered for the decision, the cousins should win. (These estimators arrive properly in notebooks 3–4;
here we use them only to answer the question.)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
candidates = [
    ("GaussianNB\n(naive, diagonal cov)", GaussianNB()),
    ("QDA\n(per-class full cov)", QuadraticDiscriminantAnalysis()),
    ("LDA\n(shared full cov)", LinearDiscriminantAnalysis()),
]
labels, vals = [], []
for name, model in candidates:
    score = cross_val_score(model, X, y, cv=cv).mean()
    labels.append(name)
    vals.append(score)
    print(f"{name.splitlines()[0]:11}: 5-fold CV accuracy {score:.4f}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.bar(
    labels, vals,
    color=[COLORS["highlight"], CLASS_CYCLE[2], CLASS_CYCLE[3]],
    edgecolor=COLORS["text"], linewidth=0.6,
)
for i, v in enumerate(vals):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", color=COLORS["text"])
ax.set_ylim(0.95, 1.0)
ax.set_ylabel("5-fold CV accuracy")
ax.set_title("Naive Bayes ties the covariance-aware models here")
plt.show()

**Read the figure.** They barely differ — and Naive Bayes (0.993) actually **ties** LDA and edges
out QDA (0.989). The model that threw away the correlation classified as well as the ones that kept it.
This is the surprise at the heart of Naive Bayes: a wrong assumption can still make a right decision
(Domingos & Pazzani, 1997). Getting the posteriors roughly *ordered* is enough — the largest one wins the
argmax, even when its exact value is off.

## Why it works here — and the honest limits

Two cautions keep this from becoming a fairy tale.

First, *why* the three tie: Naive Bayes' Gaussian form (notebook 4) is exactly **QDA with the
off-diagonal covariances set to zero** — the same model, minus the correlation terms. On these penguins
the two species barely overlap, so almost any reasonable boundary separates them and the correlation terms
have nothing left to fix. That tie is a property of **this near-separable data**, not a law — on
correlated, overlapping classes the cousins can pull ahead.

Second, surviving the *decision* is not the same as getting the *probabilities* right. Naive Bayes'
posterior numbers are often **over-confident**, precisely because it double-counts correlated evidence as
if it were independent. We measure that cost honestly on text in notebook 5.

## Your turn

1. **Predict the suffering.** Pick a different pair of penguin measurements, compute their within-class
   correlation, and predict *before testing* whether Naive Bayes would lose much accuracy on that pair.
2. **The other species.** Rebuild the real-vs-naive joint heatmaps for **Adélie** (correlation 0.33). Is
   the diagonal ridge weaker than Gentoo's? Does the difference panel agree with the smaller number?
3. **(Harder)** Explain, in your own words, how the argmax can stay correct even when the
   product-of-marginals reports the wrong probability. This is the seed of the calibration lesson in
   notebook 5.

## What you built

You met the assumption that defines Naive Bayes. You needed a **joint** likelihood for two features, saw
why estimating it directly is too expensive, and adopted the **naive** shortcut — assume the features are
**conditionally independent**, so the joint becomes a product of one-feature marginals. You **measured**
how false that is here (within-class correlation 0.33 and 0.66), then watched the decision survive it
anyway: Naive Bayes tied the covariance-aware models.

**Vocabulary**

- **joint likelihood** P(x₁, x₂ | y) — the probability of all the features at once, given the class.
- **conditional independence** — features that carry no information about each other *once the class is
  known*; the "naive" assumption.
- **the naive factorisation** — P(x₁, x₂ | y) = P(x₁ | y)·P(x₂ | y); the joint as a product of marginals.
- **what it buys / where it breaks** — d marginals instead of an exponential grid; wrong when features
  are correlated within a class.

## Going further (optional)

- **Why "naive but good"?** Domingos & Pazzani (1997) showed the assumption can be badly violated and the
  classifier still optimal under zero-one loss, because the decision needs only the *ranking* of the
  posteriors, not their exact values.
- **The probability cost.** That same robustness hides a flaw: the posteriors are poorly **calibrated**
  (over-confident). Notebook 5 measures it on text.
- **A generative model.** Naive Bayes models how the data is *generated* (P(features | class)) and inverts
  it with Bayes — the "generative" view we contrast with logistic regression in module 03.

## References

- P. Domingos, M. Pazzani (1997). *On the optimality of the simple Bayesian classifier under zero-one
  loss.* Machine Learning, 29:103–130. DOI:
  [10.1023/A:1007413511361](https://doi.org/10.1023/A:1007413511361).
- D. J. Hand, K. Yu (2001). *Idiot's Bayes — not so stupid after all?* International Statistical Review,
  69(3):385–398. DOI:
  [10.1111/j.1751-5823.2001.tb00465.x](https://doi.org/10.1111/j.1751-5823.2001.tb00465.x).
- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning*
  (§4.4.4). DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).

---

*Previous: 01 — Bayes' rule, from counts* · *Next: 03 — The Gaussian likelihood*